In [1]:
import torch
import torch.nn as nn
import torch.onnx as onnx

In [2]:
model_path = "simple_linear_model.onnx"

In [3]:
# Create a simple one layer model using a linear layer
class SimpleLinearModel(nn.Module):
    def __init__(self, input_size, output_size):
        super(SimpleLinearModel, self).__init__()
        # Define a single linear layer
        self.linear = nn.Linear(input_size, 10)
        self.linear2 = nn.Linear(10, 20)
        self.linear3 = nn.Linear(20, 15)
        self.linear4 = nn.Linear(15, output_size)

    def forward(self, x):
        # Pass input through the linear layer
        output = self.linear(x)
        output = self.linear2(output)
        output = self.linear3(output)
        output = self.linear4(output)
        return output


# Example usage
model = SimpleLinearModel(input_size=10, output_size=5)
print(model)

print("Model weights:")
for name, param in model.named_parameters():
    print(f"{name}: {param.shape}")
    if "weight" in name:
        print(f"  Weight values (first 5): {param.flatten()[:5]}")
    elif "bias" in name:
        print(f"  Bias values (first 5): {param.flatten()[:5]}")

SimpleLinearModel(
  (linear): Linear(in_features=10, out_features=10, bias=True)
  (linear2): Linear(in_features=10, out_features=20, bias=True)
  (linear3): Linear(in_features=20, out_features=15, bias=True)
  (linear4): Linear(in_features=15, out_features=5, bias=True)
)
Model weights:
linear.weight: torch.Size([10, 10])
  Weight values (first 5): tensor([-0.2207,  0.1015,  0.0715,  0.0736, -0.1903], grad_fn=<SliceBackward0>)
linear.bias: torch.Size([10])
  Bias values (first 5): tensor([-0.3079,  0.0209, -0.0449, -0.2029, -0.0360], grad_fn=<SliceBackward0>)
linear2.weight: torch.Size([20, 10])
  Weight values (first 5): tensor([-0.1581,  0.2102, -0.1822, -0.2516, -0.2445], grad_fn=<SliceBackward0>)
linear2.bias: torch.Size([20])
  Bias values (first 5): tensor([-0.2937,  0.0358, -0.1884, -0.2581, -0.2250], grad_fn=<SliceBackward0>)
linear3.weight: torch.Size([15, 20])
  Weight values (first 5): tensor([ 0.1359, -0.1308,  0.0268, -0.1328,  0.1638], grad_fn=<SliceBackward0>)
linear3.

In [4]:
# export to onnx
onnx.export(model, torch.randn(1, 10), model_path, export_params=True, opset_version=11)

In [5]:
# run the model with pytorch
input_data = torch.tensor([[1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0]])
with torch.no_grad():
    output = model(input_data)
print(output)

tensor([[-0.3799,  0.0789,  0.6093,  0.4441,  0.0036]])


In [6]:
from c_exporter.onnx_exporter import export_onnx

export_onnx(model_path, "src/model_weights.h")